# 📊 Day 10 — Week 3: Data Quality, Multi-Label, SHAP & Deployment
**Duration:** 6 Hours | **Total run time: under 60 seconds**

| Session | Time | Topic | Run Time |
|---------|------|-------|----------|
| 1 | 9:00–10:30 AM | Data Quality & Profiling | ~10 sec |
| 2 | 10:30–11:00 AM | Multi-Label Classification | ~6 sec |
| 3 | 11:00 AM–1:00 PM | SHAP Values + Online Learning | ~20 sec |
| 4 | 1:00–2:00 PM | Lunch Break | — |
| 5 | 2:00–4:00 PM | Advanced Deployment Patterns + Project | ~18 sec |

> **Zero downloads. Pure numpy + sklearn.**
---

In [ ]:
# !pip install scikit-learn matplotlib pandas numpy scipy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings, time, hashlib
from datetime import datetime
warnings.filterwarnings('ignore')

from sklearn.datasets import load_wine, load_breast_cancer, load_iris, make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier, PassiveAggressiveClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, hamming_loss,
    f1_score, classification_report, jaccard_score
)
from sklearn.inspection import permutation_importance
import sklearn
print('All imports OK')
print(f'   sklearn: {sklearn.__version__}')

---
## Session 1 — Data Quality & Profiling
**Time:** 9:00–10:30 AM | **Run time: ~10 sec**

In [ ]:
# 1.1  Generate Dirty Dataset with Realistic Issues
np.random.seed(42)
rng = np.random.default_rng(42)
N   = 600

df = pd.DataFrame({
    'age'       : rng.integers(18, 80, N).astype(float),
    'income'    : rng.exponential(50000, N).round(2),
    'score'     : rng.normal(0, 1, N).round(4),
    'tenure'    : rng.integers(0, 120, N),
    'city'      : rng.choice(['Mumbai','Delhi','Bangalore','Chennai', None], N,
                              p=[0.35, 0.30, 0.20, 0.10, 0.05]),
    'gender'    : rng.choice(['M','F','Other', None], N, p=[0.48,0.48,0.03,0.01]),
    'product'   : rng.choice(['A','B','C'], N),
})

# MNAR: high earners don't report age
df.loc[df['income'] > 80000, 'age'] = np.nan

# MAR: score missing for specific city
df.loc[(df['city'] == 'Chennai') & (rng.random(N) < 0.4), 'score'] = np.nan

# Outliers
df.loc[rng.choice(N, 5, replace=False), 'income'] = rng.uniform(500000, 2000000, 5)
df.loc[rng.choice(N, 3, replace=False), 'age']    = rng.choice([-5, 150, 999], 3)

# Duplicates
df = pd.concat([df, df.sample(15, random_state=42)], ignore_index=True)

print(f'Dataset shape: {df.shape}')
print('\nMissing values:')
print(df.isnull().sum())
print(f'\nDuplicate rows: {df.duplicated().sum()}')

In [ ]:
# 1.2  Detect MNAR: Correlation Between Missingness & Other Features
miss_df = df.isnull().astype(int)
miss_corr = miss_df.corr()

print('Missingness correlation matrix (high value = MNAR signal):')
print(miss_corr.round(3))

# Statistical test: are high-income rows more likely to have age=NaN?
from scipy import stats
high_income_mask = df['income'] > 80000
chi2, p, dof, _ = stats.chi2_contingency(
    pd.crosstab(high_income_mask, df['age'].isnull())
)
print(f'\nChi-square test (income > 80k vs age missing): p={p:.6f}')
print(f'MNAR confirmed: {p < 0.05}')

In [ ]:
# 1.3  Schema Validation Suite
class DataQualityChecker:
    def __init__(self, df):
        self.df      = df
        self.results = []

    def expect_dtype(self, col, expected_dtype):
        passed = str(self.df[col].dtype).startswith(expected_dtype)
        self.results.append({'check': f'dtype({col})', 'expected': expected_dtype,
                             'actual': str(self.df[col].dtype), 'passed': passed})
        return self

    def expect_range(self, col, lo, hi):
        vals    = self.df[col].dropna()
        passed  = (vals >= lo).all() and (vals <= hi).all()
        outlier_count = ((vals < lo) | (vals > hi)).sum()
        self.results.append({'check': f'range({col})', 'expected': f'[{lo},{hi}]',
                             'actual': f'min={vals.min():.1f} max={vals.max():.1f}',
                             'passed': passed, 'violations': int(outlier_count)})
        return self

    def expect_no_nulls(self, col):
        null_count = self.df[col].isnull().sum()
        self.results.append({'check': f'no_nulls({col})', 'expected': '0',
                             'actual': str(null_count), 'passed': null_count == 0})
        return self

    def expect_unique(self, col):
        dup = self.df.duplicated(subset=[col]).sum()
        self.results.append({'check': f'unique({col})', 'expected': '0 dupes',
                             'actual': f'{dup} dupes', 'passed': dup == 0})
        return self

    def expect_cardinality(self, col, max_vals):
        n_unique = self.df[col].nunique()
        passed   = n_unique <= max_vals
        self.results.append({'check': f'cardinality({col})', 'expected': f'<={max_vals}',
                             'actual': str(n_unique), 'passed': passed})
        return self

    def report(self):
        res_df = pd.DataFrame(self.results)
        passed = res_df['passed'].sum()
        total  = len(res_df)
        score  = passed / total
        print(f'Data Quality Score: {score:.0%}  ({passed}/{total} checks passed)')
        print(res_df[['check','expected','actual','passed']].to_string(index=False))
        return score

checker = (
    DataQualityChecker(df)
    .expect_dtype('age', 'float')
    .expect_range('age', 0, 120)
    .expect_range('income', 0, 500000)
    .expect_no_nulls('product')
    .expect_cardinality('city', 5)
    .expect_cardinality('gender', 4)
    .expect_unique('age')  # age should not be duplicated (sanity)
)
quality_score = checker.report()

In [ ]:
# 1.4  Automated Data Cleaning
def clean_dataframe(df):
    d = df.copy()
    # Remove exact duplicates
    n_before = len(d)
    d = d.drop_duplicates()
    print(f'Removed {n_before - len(d)} duplicate rows')
    # Cap outliers (Winsorize at 1st/99th percentile)
    for col in ['age','income','score','tenure']:
        lo = d[col].quantile(0.01)
        hi = d[col].quantile(0.99)
        d[col] = d[col].clip(lo, hi)
    # Impute numeric missing with median
    for col in ['age','score']:
        med = d[col].median()
        n_fill = d[col].isnull().sum()
        d[col] = d[col].fillna(med)
        print(f'Imputed {n_fill} NaN in {col} with median={med:.2f}')
    # Impute categorical with mode
    for col in ['city','gender']:
        mode_val = d[col].mode()[0]
        n_fill   = d[col].isnull().sum()
        d[col]   = d[col].fillna(mode_val)
        print(f'Imputed {n_fill} NaN in {col} with mode={mode_val}')
    return d

df_clean = clean_dataframe(df)
print(f'\nFinal shape: {df_clean.shape}')
print(f'Remaining nulls: {df_clean.isnull().sum().sum()}')

In [ ]:
# 1.5  Missingness Heatmap + Distribution Before/After
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Missingness heatmap
miss_map = df.isnull().astype(int).iloc[:100]
im = axes[0].imshow(miss_map.T, aspect='auto', cmap='RdYlGn_r', interpolation='none')
axes[0].set_yticks(range(len(df.columns)))
axes[0].set_yticklabels(df.columns)
axes[0].set_xlabel('Row index (first 100)')
axes[0].set_title('Missingness Map (yellow=missing)')
plt.colorbar(im, ax=axes[0], fraction=0.046)

# Age distribution before/after cleaning
axes[1].hist(df['age'].dropna(), bins=25, alpha=0.6, color='#D85A30', label='Raw', density=True)
axes[1].hist(df_clean['age'], bins=25, alpha=0.6, color='#1D9E75', label='Cleaned', density=True)
axes[1].set_title('Age distribution: raw vs cleaned')
axes[1].set_xlabel('Age'); axes[1].legend()

# Income outlier visualisation
axes[2].boxplot([df['income'].dropna(), df_clean['income']], labels=['Raw','Cleaned'])
axes[2].set_title('Income: outlier removal')
axes[2].set_ylabel('Income (₹)')

plt.tight_layout(); plt.show()

---
## Session 2 — Multi-Label Classification
**Time:** 10:30–11:00 AM | **Run time: ~6 sec**

In [ ]:
# 2.1  Generate Multi-Label Dataset
rng = np.random.default_rng(42)
N_ML = 800
X_ml = rng.normal(0, 1, (N_ML, 12))

# 5 labels: each depends on combinations of features
Y_ml = np.column_stack([
    (X_ml[:,0] + X_ml[:,1] + rng.normal(0, 0.4, N_ML)) > 0.3,
    (X_ml[:,2] - X_ml[:,3] + rng.normal(0, 0.4, N_ML)) > 0.1,
    (X_ml[:,4] + X_ml[:,5] + X_ml[:,6] + rng.normal(0, 0.5, N_ML)) > 0.5,
    (X_ml[:,7] * X_ml[:,8] + rng.normal(0, 0.3, N_ML)) > 0,
    (X_ml[:,9]  + rng.normal(0, 0.6, N_ML)) > 0.2,
]).astype(int)

LABEL_NAMES = ['Sports','Tech','Finance','Health','Travel']
print('Multi-label dataset:')
print(f'  Samples: {N_ML}, Labels: {Y_ml.shape[1]}')
print(f'  Label co-occurrence rate (any 2+ labels):',
      f'{(Y_ml.sum(axis=1) >= 2).mean():.1%}')
print(f'  Label frequency:')
for name, freq in zip(LABEL_NAMES, Y_ml.mean(axis=0)):
    bar = '█' * int(freq * 20)
    print(f'    {name:10s}: {freq:.2%}  {bar}')

In [ ]:
# 2.2  Binary Relevance (One Classifier Per Label)
X_tr_ml, X_te_ml, Y_tr_ml, Y_te_ml = train_test_split(
    X_ml, Y_ml, test_size=0.2, random_state=42
)

# Method 1: Binary Relevance with RandomForest
br_model = MultiOutputClassifier(
    RandomForestClassifier(60, random_state=42)
)
br_model.fit(X_tr_ml, Y_tr_ml)
Y_pred_br = br_model.predict(X_te_ml)

# Method 2: Binary Relevance with Logistic Regression
br_lr = MultiOutputClassifier(LogisticRegression(max_iter=300))
br_lr.fit(X_tr_ml, Y_tr_ml)
Y_pred_lr = br_lr.predict(X_te_ml)

print('Binary Relevance Results:')
for name, Y_pred in [('RandomForest', Y_pred_br), ('LogisticReg', Y_pred_lr)]:
    hl  = hamming_loss(Y_te_ml, Y_pred)
    mf1 = f1_score(Y_te_ml, Y_pred, average='micro')
    maf1= f1_score(Y_te_ml, Y_pred, average='macro')
    print(f'  {name:15s}: Hamming={hl:.4f}  Micro-F1={mf1:.4f}  Macro-F1={maf1:.4f}')

In [ ]:
# 2.3  Per-Label Analysis
print('Per-label metrics (RandomForest):')
print(f'{"Label":10s} | {"Precision":10s} | {"Recall":8s} | {"F1":6s} | {"Support"}')
print('-' * 55)
for i, label in enumerate(LABEL_NAMES):
    from sklearn.metrics import precision_score, recall_score
    p  = precision_score(Y_te_ml[:, i], Y_pred_br[:, i], zero_division=0)
    r  = recall_score(Y_te_ml[:, i], Y_pred_br[:, i], zero_division=0)
    f1 = f1_score(Y_te_ml[:, i], Y_pred_br[:, i], zero_division=0)
    s  = Y_te_ml[:, i].sum()
    print(f'{label:10s} | {p:10.4f} | {r:8.4f} | {f1:6.4f} | {s}')

In [ ]:
# 2.4  Label Powerset (Treat Combinations as Single Class)
# Convert Y_ml rows to integer label combination keys
def powerset_encode(Y):
    return np.array([int(''.join(map(str, row)), 2) for row in Y])

def powerset_decode(y_encoded, n_labels):
    results = []
    for val in y_encoded:
        bits = format(val, f'0{n_labels}b')
        results.append([int(b) for b in bits])
    return np.array(results)

y_ps_tr = powerset_encode(Y_tr_ml)
y_ps_te = powerset_encode(Y_te_ml)
unique_combos = len(np.unique(y_ps_tr))

print(f'Unique label combinations in train: {unique_combos}')

# Train RF on combined label
ps_model = RandomForestClassifier(60, random_state=42).fit(X_tr_ml, y_ps_tr)
y_ps_pred = ps_model.predict(X_te_ml)
Y_pred_ps = powerset_decode(y_ps_pred, Y_ml.shape[1])

hl_ps  = hamming_loss(Y_te_ml, Y_pred_ps)
mf1_ps = f1_score(Y_te_ml, Y_pred_ps, average='micro')
print(f'Label Powerset: Hamming={hl_ps:.4f}  Micro-F1={mf1_ps:.4f}')

---
## Session 3 — SHAP Values + Online Learning
**Time:** 11:00 AM–1:00 PM | **Run time: ~20 sec**

In [ ]:
# 3.1  Permutation Importance (Model-Agnostic)
wine = load_wine(as_frame=True)
X_w, y_w = wine.data.values, wine.target

rf_wine = RandomForestClassifier(100, random_state=42).fit(X_w, y_w)

pi = permutation_importance(
    rf_wine, X_w, y_w,
    n_repeats=15, random_state=42, scoring='accuracy'
)

imp_df = pd.DataFrame({
    'feature'   : wine.feature_names,
    'importance': pi.importances_mean,
    'std'       : pi.importances_std,
}).sort_values('importance', ascending=False)

print('Permutation Importance (top 8):')
print(imp_df.head(8).round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 5))
imp_sorted = imp_df.sort_values('importance', ascending=True)
ax.barh(imp_sorted['feature'], imp_sorted['importance'],
        xerr=imp_sorted['std'], color='#534AB7', alpha=0.85, capsize=3)
ax.set_title('Permutation Feature Importance — Wine Dataset')
ax.set_xlabel('Mean accuracy drop when feature shuffled')
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout(); plt.show()

In [ ]:
# 3.2  Manual SHAP Approximation (Shapley Sampling)
def approx_shap(model, X_background, x_explain, n_samples=200):
    """
    Approximate SHAP values using random coalition sampling.
    Returns mean contribution of each feature.
    """
    n_features = X_background.shape[1]
    shap_values = np.zeros(n_features)
    rng_s = np.random.default_rng(0)

    for _ in range(n_samples):
        # Random coalition: subset of features to include
        perm     = rng_s.permutation(n_features)
        bg_row   = X_background[rng_s.integers(len(X_background))].copy()

        for pos, feat in enumerate(perm):
            # With feature
            x_with              = bg_row.copy()
            x_with[perm[:pos+1]] = x_explain[perm[:pos+1]]
            # Without feature
            x_without           = bg_row.copy()
            x_without[perm[:pos]] = x_explain[perm[:pos]]

            pred_with    = model.predict_proba([x_with])[0]
            pred_without = model.predict_proba([x_without])[0]
            shap_values[feat] += (pred_with - pred_without).max()

    return shap_values / n_samples

# Explain one Wine sample
x_explain   = X_w[0]
shap_approx = approx_shap(rf_wine, X_w, x_explain, n_samples=150)

shap_df = pd.DataFrame({
    'feature'   : wine.feature_names,
    'shap_value': shap_approx
}).sort_values('shap_value', key=abs, ascending=False)

print('Approximate SHAP values for sample 0:')
print(shap_df.round(5).to_string(index=False))

In [ ]:
# 3.3  SHAP Waterfall Visualisation
fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#1D9E75' if v > 0 else '#D85A30' for v in shap_df['shap_value']]
ax.barh(shap_df['feature'], shap_df['shap_value'], color=colors, alpha=0.85)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Approximate SHAP Values — Sample 0 (Wine Dataset)')
ax.set_xlabel('SHAP value (positive = pushes toward predicted class)')
plt.tight_layout(); plt.show()

In [ ]:
# 3.4  Online Learning with SGDClassifier
X_bc, y_bc = load_breast_cancer(return_X_y=True)
scaler_bc   = StandardScaler().fit(X_bc)
X_bc_sc     = scaler_bc.transform(X_bc)

# Simulate data stream arriving in mini-batches
BATCH_SIZE = 20
sgd = SGDClassifier(loss='log_loss', random_state=42, max_iter=1)
pac = PassiveAggressiveClassifier(random_state=42, max_iter=1)

batch_accs_sgd, batch_accs_pac = [], []
all_preds_sgd, all_preds_pac   = [], []
all_true = []

for start in range(0, len(X_bc_sc)-BATCH_SIZE, BATCH_SIZE):
    X_batch = X_bc_sc[start:start+BATCH_SIZE]
    y_batch = y_bc[start:start+BATCH_SIZE]

    sgd.partial_fit(X_batch, y_batch, classes=[0, 1])
    pac.partial_fit(X_batch, y_batch, classes=[0, 1])

    if start > 0:
        preds_sgd = sgd.predict(X_bc_sc[:start])
        preds_pac = pac.predict(X_bc_sc[:start])
        batch_accs_sgd.append(accuracy_score(y_bc[:start], preds_sgd))
        batch_accs_pac.append(accuracy_score(y_bc[:start], preds_pac))

print(f'SGD final accuracy : {batch_accs_sgd[-1]:.4f}')
print(f'PAC final accuracy : {batch_accs_pac[-1]:.4f}')

fig, ax = plt.subplots(figsize=(10, 4))
batches = range(len(batch_accs_sgd))
ax.plot(batches, batch_accs_sgd, color='#534AB7', linewidth=1.8, label='SGDClassifier')
ax.plot(batches, batch_accs_pac, color='#1D9E75', linewidth=1.8, label='PassiveAggressive')
ax.set_xlabel('Batch number'); ax.set_ylabel('Cumulative accuracy')
ax.set_title('Online Learning Convergence (mini-batch size=20)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## Lunch Break — 1:00–2:00 PM
1. What makes MNAR different from MAR — why is it dangerous for imputation?
2. When would Label Powerset beat Binary Relevance for multi-label?
3. What does permutation importance measure that built-in RF importance does not?
4. When would you use PassiveAggressive over SGD for online learning?
---

## Session 5 — Advanced Deployment Patterns + Project
**Time:** 2:00–4:00 PM | **Run time: ~18 sec**

In [ ]:
# 5.1  Blue-Green + Canary Router
class CanaryRouter:
    """Route a percentage of traffic to the new (green) model."""
    def __init__(self, blue_model, green_model, canary_pct=0.10):
        self.blue       = blue_model
        self.green      = green_model
        self.canary_pct = canary_pct
        self.counts     = {'blue': 0, 'green': 0}
        self.green_preds, self.blue_preds = [], []

    def predict(self, X):
        if np.random.rand() < self.canary_pct:
            self.counts['green'] += 1
            pred = self.green.predict_proba(X)
            self.green_preds.append(pred[0, 1])
            return pred, 'green'
        self.counts['blue'] += 1
        pred = self.blue.predict_proba(X)
        self.blue_preds.append(pred[0, 1])
        return pred, 'blue'

    def promote_green(self):
        self.canary_pct = 1.0
        print('Green model promoted to 100% champion')

    def rollback(self):
        self.canary_pct = 0.0
        print('Rolled back to blue (old) model')

    def traffic_report(self):
        total = self.counts['blue'] + self.counts['green']
        return {
            'blue_pct'  : self.counts['blue']  / total,
            'green_pct' : self.counts['green'] / total,
            'total_req' : total,
        }

# Setup models
X_bc, y_bc = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X_bc, y_bc, test_size=0.2, random_state=42)

blue_model  = RandomForestClassifier(50, random_state=0).fit(X_tr, y_tr)
green_model = GradientBoostingClassifier(80, random_state=42).fit(X_tr, y_tr)

router = CanaryRouter(blue_model, green_model, canary_pct=0.10)
np.random.seed(0)
for i in range(2000):
    router.predict(X_te[[i % len(X_te)]])

report = router.traffic_report()
print('Canary Router Traffic Report:')
for k, v in report.items():
    print(f'  {k:12s}: {v:.2%}' if isinstance(v, float) else f'  {k:12s}: {v}')

print(f'\nBlue  AUC: {roc_auc_score(y_te, blue_model.predict_proba(X_te)[:,1]):.4f}')
print(f'Green AUC: {roc_auc_score(y_te, green_model.predict_proba(X_te)[:,1]):.4f}')

In [ ]:
# 5.2  Shadow Mode Deployment
class ShadowDeployment:
    """Run new model in shadow — compare outputs without affecting users."""
    def __init__(self, champion, shadow):
        self.champion = champion
        self.shadow   = shadow
        self.divergences = []

    def predict(self, X):
        champ_prob  = self.champion.predict_proba(X)[0, 1]
        shadow_prob = self.shadow.predict_proba(X)[0, 1]
        divergence  = abs(champ_prob - shadow_prob)
        self.divergences.append(divergence)
        return champ_prob   # only champion result served to user

    def divergence_report(self):
        d = np.array(self.divergences)
        return {
            'mean_divergence' : round(float(d.mean()), 5),
            'max_divergence'  : round(float(d.max()), 5),
            'pct_agree'       : round(float((d < 0.1).mean()), 4),
            'n_predictions'   : len(d),
        }

shadow = ShadowDeployment(champion=blue_model, shadow=green_model)
for i in range(500):
    shadow.predict(X_te[[i % len(X_te)]])

print('Shadow Mode Divergence Report:')
for k, v in shadow.divergence_report().items():
    print(f'  {k:20s}: {v}')

In [ ]:
# 5.3  Prediction Cache + Input Validator
class PredictionCache:
    """LRU-style cache for identical inputs."""
    def __init__(self, model, max_size=500):
        self.model    = model
        self.cache    = {}
        self.hits     = 0
        self.misses   = 0
        self.max_size = max_size

    def _key(self, X):
        return hashlib.md5(X.tobytes()).hexdigest()

    def predict(self, X):
        key = self._key(X)
        if key in self.cache:
            self.hits += 1
            return self.cache[key]
        self.misses += 1
        result = self.model.predict_proba(X)[0, 1]
        if len(self.cache) >= self.max_size:
            self.cache.pop(next(iter(self.cache)))  # evict oldest
        self.cache[key] = result
        return result

    def stats(self):
        total = self.hits + self.misses
        return {'hit_rate': self.hits/total if total else 0,
                'cache_size': len(self.cache), 'total': total}

# Simulate repeated requests (30% are cache hits)
cache = PredictionCache(blue_model)
np.random.seed(1)
for _ in range(1000):
    idx = np.random.choice(np.arange(10))  # only 10 unique queries → high hit rate
    cache.predict(X_te[[idx]])

stats = cache.stats()
print(f'Cache hit rate   : {stats["hit_rate"]:.1%}')
print(f'Cache size       : {stats["cache_size"]}')
print(f'Total requests   : {stats["total"]}')

In [ ]:
# 5.4  Week 3 Kickoff Project: Multi-Label Tag Prediction System
print('=== MULTI-LABEL TAG PREDICTION SYSTEM ===')
print('Scenario: Predict article tags from text features\n')

# Reuse multi-label dataset from session 2
# Full pipeline: data → clean → train → evaluate → deploy with canary

# Step 1: Feature scaling
scaler_ml = StandardScaler().fit(X_tr_ml)
X_tr_sc   = scaler_ml.transform(X_tr_ml)
X_te_sc_ml= scaler_ml.transform(X_te_ml)

# Step 2: Train two multi-label models
models_ml = {
    'RF-BinaryRelevance'  : MultiOutputClassifier(RandomForestClassifier(60, random_state=42)),
    'GB-BinaryRelevance'  : MultiOutputClassifier(GradientBoostingClassifier(50, random_state=42)),
}

leaderboard_ml = []
for name, model in models_ml.items():
    model.fit(X_tr_sc, Y_tr_ml)
    Y_pred = model.predict(X_te_sc_ml)
    hl     = hamming_loss(Y_te_ml, Y_pred)
    mf1    = f1_score(Y_te_ml, Y_pred, average='micro')
    maf1   = f1_score(Y_te_ml, Y_pred, average='macro')
    leaderboard_ml.append({'Model':name,'Hamming':round(hl,4),'Micro-F1':round(mf1,4),'Macro-F1':round(maf1,4)})
    print(f'{name}: Hamming={hl:.4f}  Micro-F1={mf1:.4f}  Macro-F1={maf1:.4f}')

print()
print('Per-label F1 scores (best model):')
best_ml = models_ml['RF-BinaryRelevance']
Y_pred_best = best_ml.predict(X_te_sc_ml)
for i, label in enumerate(LABEL_NAMES):
    f1_l = f1_score(Y_te_ml[:,i], Y_pred_best[:,i])
    print(f'  {label:10s}: {f1_l:.4f}  {"█"*int(f1_l*20)}')

---
## Day 10 Summary

| Concept | Skill Gained |
|---|---|
| MCAR/MAR/MNAR | Missingness types, chi-square test for MNAR |
| Missingness heatmap | Co-occurrence patterns, visual profiling |
| Schema validation | DataQualityChecker with 5 assertion types |
| Automated cleaning | Dedup, Winsorise, median/mode imputation |
| Binary Relevance | MultiOutputClassifier, one RF per label |
| Label Powerset | Encode combinations as single-class problem |
| Hamming loss | Per-label error rate interpretation |
| Permutation importance | Model-agnostic, captures interactions |
| Manual SHAP | Shapley coalition sampling approximation |
| Online learning | SGDClassifier.partial_fit, PassiveAggressive |
| Canary router | Traffic percentage split, promote/rollback |
| Shadow deployment | Compare models without user impact |
| Prediction cache | MD5 hash key, hit rate measurement |

---
## Day 11 Preview
- Causal inference basics: correlation vs causation
- Survival analysis: time-to-event modelling
- Multi-armed bandit: ε-greedy and Thompson sampling
- Model compression: pruning and quantisation concepts
- Final capstone: production-grade ML system design

In [ ]:
# Bonus: Day 10 in one cell
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import hamming_loss, f1_score
rng = np.random.default_rng(0)
Xb  = rng.normal(0, 1, (400, 8))
Yb  = np.column_stack([(Xb[:,i]+rng.normal(0,.5,400))>0 for i in range(3)]).astype(int)
Xbt, Xbv, Ybt, Ybv = train_test_split(Xb, Yb, test_size=0.2, random_state=42)
m   = MultiOutputClassifier(RandomForestClassifier(30, random_state=42)).fit(Xbt, Ybt)
Yp  = m.predict(Xbv)
print(f'Quick multi-label: Hamming={hamming_loss(Ybv,Yp):.4f} Micro-F1={f1_score(Ybv,Yp,average="micro"):.4f}')
print(f'Canary: blue={report["blue_pct"]:.0%} green={report["green_pct"]:.0%}')
print(f'Cache hit rate: {stats["hit_rate"]:.0%}')
print('\nDay 10 complete — data quality, multi-label, SHAP, online learning, deployment patterns!')